# Shared team streams, graph, possession memory and span supervision

Four predeclared ablations keep the original audio cache and supplied labels.
Five whole-window folds use seed 0; final models average seeds 0/1/2, 60 epochs.
All fitting precedes main validation scoring. Report fixed decoder cost 8 and
the fixed 1/2/4/8 grid chosen using training-window OOF labels only.
No pretrained backbone replacements, new annotations or paid compute.


In [ ]:
from pathlib import Path
import hashlib,json,os,subprocess,sys,time
import torch
assert torch.cuda.is_available()
INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working/possession_relational')
WORK.mkdir(parents=True,exist_ok=True)
print('GPU:',torch.cuda.get_device_name(0),flush=True)
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
roots=[p.parent for p in INPUT.rglob('RELATIONAL_EXPERIMENT.md') if (p.parent/'relational_possession').is_dir()]
assert len(roots)==1
CODE=roots[0]
configs=list(INPUT.rglob('relational_input.json'))
assert len(configs)==1
LABELS=configs[0].parent
plan=json.loads(configs[0].read_text())
base=[p for p in INPUT.rglob('features.npz') if sha(p)==plan['feature_sha256']]
audio=[p for p in INPUT.rglob('audio_cache.npz') if sha(p)==plan['audio_sha256']]
assert len(base)==len(audio)==1
FEATURES,AUDIO,GRAPH=base[0],audio[0],LABELS/'graph.npz'
assert sha(GRAPH)==plan['graph_sha256']
for name in ('train','validation'):
    assert sha(LABELS/f'{name}.csv')==plan[f'{name}_sha256']
prior=[p for p in INPUT.rglob('baseline.csv') if sha(p)==plan['comparison_sha256']]
assert len(prior)==1
PRIOR=prior[0]
DEPS=Path('/tmp/barca_relational_python')
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check',
               '--target',str(DEPS),'-r',str(CODE/'requirements-temporal.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check',
               '--target',str(DEPS),'--no-deps','-r',str(CODE/'requirements-foundation.txt')],check=True)
env=dict(os.environ,PYTHONPATH=str(DEPS)+os.pathsep+str(CODE),HF_HUB_OFFLINE='1',
         PYTHONDONTWRITEBYTECODE='1',CUBLAS_WORKSPACE_CONFIG=':4096:8')
def run(*args):subprocess.run([sys.executable,*map(str,args)],cwd=CODE,env=env,check=True)
run('-m','unittest','discover','-s','tests','-v')
print('Full test suite complete, starting fixed real-data architecture study.',flush=True)


In [ ]:
RUN=WORK/'study'
started=time.monotonic()
run('-m','relational_possession','compare','--features',FEATURES,'--audio',AUDIO,'--graph',GRAPH,
    '--train-labels',LABELS/'train.csv','--validation-labels',LABELS/'validation.csv',
    '--audit',LABELS/'annotation_audit.json','--out',RUN,'--device','cuda','--epochs','60',
    '--seeds','0','1','2','--cross-validate','--calibrate','--bootstrap','1000','--compare',PRIOR)


In [ ]:
arms=('symmetric_tcn','graph_tcn','graph_memory','graph_memory_span')
checks={}
for arm in arms:
    for calibrated in (False,True):
        name='calibrated.csv' if calibrated else 'predictions.csv'
        output=RUN/'global'/arm/('regenerated_'+name)
        args=['-m','relational_possession','predict','--features',FEATURES,'--audio',AUDIO,
              '--graph',GRAPH,'--model',RUN/'global'/arm,'--out',output,'--device','cuda']
        if calibrated:args.append('--calibrated')
        run(*args)
        assert output.read_bytes()==(RUN/'global'/arm/name).read_bytes()
        checks[arm+('_calibrated' if calibrated else '')]=sha(output)
        print('Byte-identical relational model regeneration:',arm,name,flush=True)
source={str(p.relative_to(CODE)):sha(p) for root in ('relational_possession','temporal_possession','learned_possession','structured_possession') for p in (CODE/root).glob('*.py')}
protocol=dict(torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),
              elapsed_including_regeneration_seconds=time.monotonic()-started,
              all_eight_outputs_regenerated=True,regeneration=checks,source_sha256=source,
              audio_encoder_unchanged=True,references_unchanged=True,private_inputs=plan)
(WORK/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')
print('RELATIONAL GPU RUN COMPLETE',flush=True)
